# ESM-C SAE Layer Ensemble Analysis

This notebook investigates ways to combine multiple fixed DeltaEmbSAE model/layer outputs for BRCA1 while excluding ESM2 entirely. It reuses the artifacts produced by `lower_context.ipynb`, including the layer-level Spearman/AUC table, fixed SAE reconstruction table, and per-layer fitness CSVs.

Analyses covered here:

- Rank ensembles over ESM-C SAE layers using median, best, and worst rank.
- Collections: all ESM-C SAE layers, top AUC, top MaveDB Spearman, top SAE reconstruction R2, and a balanced AUC/Spearman/reconstruction subset.
- PCA-concatenated SAE inference: per selected SAE, fit PCA to SAE activations, keep the globally most important PCs up to `PCA_MAX_COMPONENTS`, concatenate those coordinates, run popDMS inference, and save the gamma consistency plot.

I interpret "MaveDB reconstruction" as the SAE reconstruction R2 computed on the MaveDB variant feature table, and also include a top MaveDB-Spearman collection so both readings are available.

In [ ]:
from pathlib import Path
import os
import pickle
import subprocess
import sys
import importlib

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, Markdown, display

import lower_context_helpers as h
h = importlib.reload(h)

sns.set_theme(style="darkgrid")
paths = h.default_brca1_paths(REPO_ROOT)
paths

## Configuration

`SUBMIT_JOB` is `False` by default, so running this notebook writes the payload and Slurm script without submitting. Set it to `True` after reviewing `tasks_df`. PCA jobs are the expensive part; the rank ensembles are lightweight but are included in the same array for reproducibility.

In [ ]:
DATASET_NAME = "BRCA1"
EMBEDDING_TYPE = "max_pool"
FIGURE_DPI = 450

OUTPUT_ROOT = paths.analysis_dir / "esmc_sae_layer_ensemble_analysis"
JOB_ROOT = paths.job_dir / "esmc_sae_layer_ensemble_analysis"
for directory in [OUTPUT_ROOT, JOB_ROOT, OUTPUT_ROOT / "metrics", OUTPUT_ROOT / "fitness", OUTPUT_ROOT / "figures"]:
    directory.mkdir(parents=True, exist_ok=True)

COLLECTION_TOP_N = 12
BALANCED_PER_MODEL = 4
RANK_AGGREGATIONS = ["rank_median", "rank_worst", "rank_best"]

PCA_MAX_COMPONENTS = 1000
PCA_PER_SOURCE_COMPONENTS = 64
PCA_STANDARDIZE_INPUTS = True
PCA_IMPORTANCE_METRIC = "explained_variance_ratio"
PCA_RANDOM_SEED = 42

CREATE_JOB = True
SUBMIT_JOB = False
CREATE_MAVEDB_GAMMA_JOB = True
SUBMIT_MAVEDB_GAMMA_JOB = False
MAX_ACTIVE_TASKS = 4
PYTHON_EXECUTABLE = "python3"
SLURM_PARTITION = "any_cpu"
SLURM_CPUS_PER_TASK = 4
SLURM_MEM = "48G"
SLURM_TIME = "12:00:00"

RUN_LOCAL_SMOKE_TASK = False
LOCAL_SMOKE_TASK_IDX = 0

print(f"Output root: {OUTPUT_ROOT}")
print(f"Gamma diagnostics plots will be under: {OUTPUT_ROOT / 'pca_concat'}")


## Load Existing Tables And Build ESM-C Collections

The helper filters source rows to `biohub/ESMC-*` / `ESMC-*` rows only and excludes layer 0. If ESM-C 6B artifacts are added later, they will be included automatically once they appear in the metrics tables.

In [ ]:
runner = h.processed_brca1_runner(paths, model_name="biohub/ESMC-300M", dataset_name=DATASET_NAME, embedding_type=EMBEDDING_TYPE)
metrics_df = h.load_metrics_table(paths.spearman_auc_table_path)
sae_metrics_df = pd.read_csv(paths.fixed_sae_metrics_path) if paths.fixed_sae_metrics_path.is_file() else pd.DataFrame()

collections = h.build_esmc_sae_collections(
    metrics_df,
    sae_metrics_df,
    embedding_type=EMBEDDING_TYPE,
    top_n=COLLECTION_TOP_N,
    balanced_per_model=BALANCED_PER_MODEL,
)
assert collections, "No ESM-C SAE collections were found. Run/collect lower_context.ipynb first."
assert all(h.is_esmc_row(rows).all() for rows in collections.values()), "A non-ESM-C row leaked into an ESM-C collection."

collection_summary = pd.DataFrame([
    {
        "collection_name": name,
        "n_layers": len(rows),
        "models": ", ".join(sorted(rows["model_short"].dropna().astype(str).unique())),
        "layer_min": int(rows["layer"].min()) if len(rows) else np.nan,
        "layer_max": int(rows["layer"].max()) if len(rows) else np.nan,
        "mean_auc": rows["auc"].mean(),
        "mean_spearman": rows["spearman_rho"].mean(),
        "mean_reconstruction_r2": rows.get("reconstruction_r2", pd.Series(dtype=float)).mean(),
    }
    for name, rows in collections.items()
])
collection_summary.to_csv(OUTPUT_ROOT / "esmc_sae_collection_summary.csv", index=False)
display(collection_summary)

In [ ]:
preview_rows = []
for name, rows in collections.items():
    preview = rows.sort_values(["auc", "spearman_rho"], ascending=[False, False]).head(8).copy()
    preview.insert(0, "collection_name", name)
    preview_rows.append(preview)
collection_preview_df = pd.concat(preview_rows, ignore_index=True, sort=False)
display(
    collection_preview_df[
        ["collection_name", "model_short", "layer", "auc", "spearman_rho", "reconstruction_r2", "feature_path", "fitness_path"]
    ].assign(
        auc=lambda df: pd.to_numeric(df["auc"], errors="coerce").round(3),
        spearman_rho=lambda df: pd.to_numeric(df["spearman_rho"], errors="coerce").round(3),
        reconstruction_r2=lambda df: pd.to_numeric(df["reconstruction_r2"], errors="coerce").round(3),
    )
)

## Build Slurm Tasks

Rank tasks cover every collection and each requested rank aggregation. PCA-concat tasks run once per collection and save the cross-replicate gamma consistency plot for that inferred concatenated feature space.

In [ ]:
def _pretty_collection_label(name):
    return name.replace("_", " ").replace("top", "top ")

rank_collections = list(collections)
pca_collections = [
    "all_esmc_sae_layers",
    f"best_auc_top{COLLECTION_TOP_N}",
    f"best_mavedb_spearman_top{COLLECTION_TOP_N}",
    f"best_reconstruction_r2_top{COLLECTION_TOP_N}",
    f"balanced_auc_spearman_reconstruction_top{BALANCED_PER_MODEL}_per_model",
]
pca_collections = [name for name in pca_collections if name in collections]

tasks = []
for collection_name in rank_collections:
    for aggregation in RANK_AGGREGATIONS:
        label = f"{_pretty_collection_label(collection_name)} {aggregation.replace('rank_', '')} rank"
        tasks.append({
            "task_type": "rank_ensemble",
            "collection_name": collection_name,
            "aggregation": aggregation,
            "method_label": label,
            "short_label": label.replace("all esmc sae layers", "all ESMC"),
        })

crossrep_pca_tasks = []
for collection_name in pca_collections:
    label = f"{_pretty_collection_label(collection_name)} PCA concat"
    crossrep_pca_tasks.append({
        "task_type": "pca_concat",
        "collection_name": collection_name,
        "method_label": f"{label} (consistency gamma)",
        "short_label": f"{label.replace('all esmc sae layers', 'all ESMC')} consistency gamma",
        "max_components": PCA_MAX_COMPONENTS,
        "per_source_components": PCA_PER_SOURCE_COMPONENTS,
        "standardize": PCA_STANDARDIZE_INPUTS,
        "importance_metric": PCA_IMPORTANCE_METRIC,
        "seed": PCA_RANDOM_SEED,
        "gamma_selection_objective": "cross_replicate_consistency",
    })
tasks.extend(crossrep_pca_tasks)

mavedb_gamma_tasks = []
for collection_name in pca_collections:
    label = f"{_pretty_collection_label(collection_name)} PCA concat"
    mavedb_gamma_tasks.append({
        "task_type": "pca_concat",
        "collection_name": collection_name,
        "method_label": f"{label} (MaveDB gamma)",
        "short_label": f"{label.replace('all esmc sae layers', 'all ESMC')} MaveDB gamma",
        "max_components": PCA_MAX_COMPONENTS,
        "per_source_components": PCA_PER_SOURCE_COMPONENTS,
        "standardize": PCA_STANDARDIZE_INPUTS,
        "importance_metric": PCA_IMPORTANCE_METRIC,
        "seed": PCA_RANDOM_SEED,
        "gamma_selection_objective": "mavedb_spearman",
    })

tasks_df = pd.DataFrame(tasks)
tasks_df.insert(0, "task_idx", range(len(tasks_df)))
tasks_df["n_source_layers"] = tasks_df["collection_name"].map(lambda name: len(collections[name]))
tasks_df.to_csv(OUTPUT_ROOT / "esmc_sae_ensemble_tasks.csv", index=False)

mavedb_gamma_tasks_df = pd.DataFrame(mavedb_gamma_tasks)
mavedb_gamma_tasks_df.insert(0, "task_idx", range(len(mavedb_gamma_tasks_df)))
mavedb_gamma_tasks_df["n_source_layers"] = mavedb_gamma_tasks_df["collection_name"].map(lambda name: len(collections[name]))
mavedb_gamma_tasks_df.to_csv(OUTPUT_ROOT / "esmc_sae_ensemble_mavedb_gamma_tasks.csv", index=False)

display(tasks_df)
display(mavedb_gamma_tasks_df)


## Write Payload And Slurm Script

The generated worker writes one status JSON per task under `OUTPUT_ROOT/status`. Full task outputs land in `OUTPUT_ROOT/metrics`, `OUTPUT_ROOT/fitness`, and `OUTPUT_ROOT/pca_concat`.

In [ ]:
payload = {
    "repo_root": str(REPO_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "dataset_name": DATASET_NAME,
    "embedding_type": EMBEDDING_TYPE,
    "metrics_path": str(paths.spearman_auc_table_path),
    "sae_metrics_path": str(paths.fixed_sae_metrics_path),
    "clinvar_annotation_cache_path": str(paths.clinvar_annotation_cache_path),
    "primary_key": "hgvs_nt",
    "score_col": "score",
    "collection_top_n": COLLECTION_TOP_N,
    "balanced_per_model": BALANCED_PER_MODEL,
    "max_components": PCA_MAX_COMPONENTS,
    "per_source_components": PCA_PER_SOURCE_COMPONENTS,
    "standardize_pca_inputs": PCA_STANDARDIZE_INPUTS,
    "importance_metric": PCA_IMPORTANCE_METRIC,
    "seed": PCA_RANDOM_SEED,
    "tasks": tasks,
}

payload_path = JOB_ROOT / "esmc_sae_layer_ensemble_payload.pkl"
script_path = JOB_ROOT / "submit_esmc_sae_layer_ensemble_array.sh"
log_dir = JOB_ROOT / "logs"

if CREATE_JOB:
    with payload_path.open("wb") as handle:
        pickle.dump(payload, handle)
    h.write_slurm_array_job(
        script_path=script_path,
        payload_path=payload_path,
        runner_script=REPO_ROOT / "job_scripts" / "run_esmc_sae_ensemble_task.py",
        n_tasks=len(tasks),
        job_name="brca1_esmc_sae_ens",
        repo_root=REPO_ROOT,
        log_dir=log_dir,
        partition=SLURM_PARTITION,
        cpus_per_task=SLURM_CPUS_PER_TASK,
        mem=SLURM_MEM,
        time=SLURM_TIME,
        max_active_tasks=MAX_ACTIVE_TASKS,
        python_executable=PYTHON_EXECUTABLE,
    )
    print(f"Cross-replicate payload: {payload_path}")
    print(f"Cross-replicate Slurm script: {script_path}")
    if SUBMIT_JOB:
        completed = subprocess.run(["sbatch", str(script_path)], check=True, capture_output=True, text=True)
        print(completed.stdout.strip())

mavedb_gamma_payload = dict(payload)
mavedb_gamma_payload["tasks"] = mavedb_gamma_tasks
mavedb_gamma_payload["gamma_selection_objective"] = "mavedb_spearman"
mavedb_gamma_payload_path = JOB_ROOT / "esmc_sae_layer_ensemble_mavedb_gamma_payload.pkl"
mavedb_gamma_script_path = JOB_ROOT / "submit_esmc_sae_layer_ensemble_mavedb_gamma_array.sh"

if CREATE_MAVEDB_GAMMA_JOB:
    with mavedb_gamma_payload_path.open("wb") as handle:
        pickle.dump(mavedb_gamma_payload, handle)
    h.write_slurm_array_job(
        script_path=mavedb_gamma_script_path,
        payload_path=mavedb_gamma_payload_path,
        runner_script=REPO_ROOT / "job_scripts" / "run_esmc_sae_ensemble_task.py",
        n_tasks=len(mavedb_gamma_tasks),
        job_name="brca1_esmc_sae_mvg",
        repo_root=REPO_ROOT,
        log_dir=JOB_ROOT / "logs_mavedb_gamma",
        partition=SLURM_PARTITION,
        cpus_per_task=SLURM_CPUS_PER_TASK,
        mem=SLURM_MEM,
        time=SLURM_TIME,
        max_active_tasks=MAX_ACTIVE_TASKS,
        python_executable=PYTHON_EXECUTABLE,
    )
    print(f"MaveDB-gamma payload: {mavedb_gamma_payload_path}")
    print(f"MaveDB-gamma Slurm script: {mavedb_gamma_script_path}")
    if SUBMIT_MAVEDB_GAMMA_JOB:
        completed = subprocess.run(["sbatch", str(mavedb_gamma_script_path)], check=True, capture_output=True, text=True)
        print(completed.stdout.strip())


In [ ]:
if RUN_LOCAL_SMOKE_TASK:
    completed = subprocess.run(
        [PYTHON_EXECUTABLE, str(REPO_ROOT / "job_scripts" / "run_esmc_sae_ensemble_task.py"), str(payload_path), str(LOCAL_SMOKE_TASK_IDX)],
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout)

## Collect Finished Task Outputs

Run this section after the Slurm array has finished. It combines every per-task metrics table and writes a single review-star metrics table for downstream plotting.

In [ ]:
metrics_files = sorted((OUTPUT_ROOT / "metrics").glob("*_metrics_by_review_stars.csv"))
print(f"Found {len(metrics_files)} metrics file(s).")
if metrics_files:
    esmc_ensemble_metrics_df = pd.concat([pd.read_csv(path) for path in metrics_files], ignore_index=True, sort=False)
    for column in ["gamma_opt", "gamma_plot_path", "gamma_table_path", "gamma_pair_table_path", "gamma_selection_objective", "selected_gamma_mean_pairwise_pearson_r", "selected_gamma_mavedb_spearman", "selected_gamma_grid", "n_selected_components"]:
        if column not in esmc_ensemble_metrics_df.columns:
            esmc_ensemble_metrics_df[column] = np.nan
    def _normalize_gamma_objective(row):
        value = row.get("gamma_selection_objective", "")
        is_blank = pd.isna(value) or str(value).strip() == ""
        if row.get("aggregation") == "pca_concat" and is_blank:
            return "cross_replicate_consistency"
        if is_blank:
            return "not_applicable"
        return value

    esmc_ensemble_metrics_df["gamma_selection_objective"] = esmc_ensemble_metrics_df.apply(_normalize_gamma_objective, axis=1)

    def _gamma_objective_label(row):
        label = str(row["method_label"])
        if row.get("aggregation") != "pca_concat":
            return label
        objective = row.get("gamma_selection_objective")
        if objective == "cross_replicate_consistency" and "gamma" not in label.lower():
            return f"{label} (consistency gamma)"
        if objective == "mavedb_spearman" and "mavedb" not in label.lower():
            return f"{label} (MaveDB gamma)"
        return label

    esmc_ensemble_metrics_df["method_label"] = esmc_ensemble_metrics_df.apply(_gamma_objective_label, axis=1)
    if "short_label" in esmc_ensemble_metrics_df.columns:
        esmc_ensemble_metrics_df["short_label"] = esmc_ensemble_metrics_df.apply(
            lambda row: row["method_label"] if row.get("aggregation") == "pca_concat" else row.get("short_label", row["method_label"]),
            axis=1,
        )
    combined_metrics_path = paths.table_dir / "BRCA1_esmc_sae_layer_ensemble_metrics_by_review_stars.csv"
    esmc_ensemble_metrics_df.to_csv(combined_metrics_path, index=False)
    display(
        esmc_ensemble_metrics_df[
            ["review_filter", "method_label", "aggregation", "model_filter", "gamma_selection_objective", "n_component_model_layers", "spearman_rho", "auc", "functional_score_auc", "gamma_opt", "selected_gamma_mean_pairwise_pearson_r", "selected_gamma_mavedb_spearman", "gamma_plot_path"]
        ].assign(
            spearman_rho=lambda df: pd.to_numeric(df["spearman_rho"], errors="coerce").round(3),
            auc=lambda df: pd.to_numeric(df["auc"], errors="coerce").round(3),
            functional_score_auc=lambda df: pd.to_numeric(df["functional_score_auc"], errors="coerce").round(3),
            selected_gamma_mean_pairwise_pearson_r=lambda df: pd.to_numeric(df["selected_gamma_mean_pairwise_pearson_r"], errors="coerce").round(3),
            selected_gamma_mavedb_spearman=lambda df: pd.to_numeric(df["selected_gamma_mavedb_spearman"], errors="coerce").round(3),
        )
    )
else:
    esmc_ensemble_metrics_df = pd.DataFrame()
    combined_metrics_path = None


## Performance Summary

This table collapses the review-star rows into one row per method/objective and keeps the all-binary MaveDB Spearman plus AUC at all, 1, 2, and 3-star filters.


In [ ]:
if not esmc_ensemble_metrics_df.empty:
    performance_summary_df = h.performance_summary_table(esmc_ensemble_metrics_df)
    performance_summary_path = paths.table_dir / "BRCA1_esmc_sae_layer_ensemble_performance_summary.csv"
    performance_summary_df.to_csv(performance_summary_path, index=False)
    display_columns = [
        "spearman_auc_rank",
        "spearman_auc_rank_score",
        "mavedb_spearman_rank",
        "auc_all_binary_rank",
        "method_label",
        "aggregation",
        "model_filter",
        "gamma_selection_objective",
        "n_component_model_layers",
        "n_selected_components",
        "gamma_opt",
        "selected_gamma_mean_pairwise_pearson_r",
        "selected_gamma_mavedb_spearman",
        "mavedb_spearman",
        "auc_all_binary",
        "auc_min_1_stars",
        "auc_min_2_stars",
        "auc_min_3_stars",
    ]
    display(
        performance_summary_df[[col for col in display_columns if col in performance_summary_df.columns]].assign(
            spearman_auc_rank_score=lambda df: pd.to_numeric(df.get("spearman_auc_rank_score", np.nan), errors="coerce").round(2),
            mavedb_spearman_rank=lambda df: pd.to_numeric(df.get("mavedb_spearman_rank", np.nan), errors="coerce").astype("Int64"),
            auc_all_binary_rank=lambda df: pd.to_numeric(df.get("auc_all_binary_rank", np.nan), errors="coerce").astype("Int64"),
            gamma_opt=lambda df: pd.to_numeric(df.get("gamma_opt", np.nan), errors="coerce"),
            selected_gamma_mean_pairwise_pearson_r=lambda df: pd.to_numeric(df.get("selected_gamma_mean_pairwise_pearson_r", np.nan), errors="coerce").round(3),
            selected_gamma_mavedb_spearman=lambda df: pd.to_numeric(df.get("selected_gamma_mavedb_spearman", np.nan), errors="coerce").round(3),
            mavedb_spearman=lambda df: pd.to_numeric(df.get("mavedb_spearman", np.nan), errors="coerce").round(3),
            auc_all_binary=lambda df: pd.to_numeric(df.get("auc_all_binary", np.nan), errors="coerce").round(3),
            auc_min_1_stars=lambda df: pd.to_numeric(df.get("auc_min_1_stars", np.nan), errors="coerce").round(3),
            auc_min_2_stars=lambda df: pd.to_numeric(df.get("auc_min_2_stars", np.nan), errors="coerce").round(3),
            auc_min_3_stars=lambda df: pd.to_numeric(df.get("auc_min_3_stars", np.nan), errors="coerce").round(3),
        )
    )
    print(f"Performance summary ranked by combined MaveDB-Spearman/AUC: {performance_summary_path}")
else:
    performance_summary_df = pd.DataFrame()


## Gamma Objective Comparison

After the MaveDB-gamma jobs finish, this compares the selected-gamma replicate consistency, all-binary MaveDB Spearman, and all-binary ClinVar AUC against the cross-replicate gamma jobs for the same PCA collection.


In [ ]:
if not performance_summary_df.empty:
    gamma_objective_comparison_df = h.gamma_objective_comparison_table(performance_summary_df)
    gamma_objective_comparison_path = paths.table_dir / "BRCA1_esmc_sae_layer_ensemble_gamma_objective_comparison.csv"
    gamma_objective_comparison_df.to_csv(gamma_objective_comparison_path, index=False)
    display(gamma_objective_comparison_df)
    gamma_objective_comparison_figure_path = paths.figure_dir / "BRCA1_esmc_sae_layer_ensemble_gamma_objective_comparison.png"
    print(f"Gamma objective comparison table: {gamma_objective_comparison_path}")
else:
    gamma_objective_comparison_df = pd.DataFrame()


## Best Worst-Rank Ensemble Gamma Sweep

This writes a single cluster job for the best MaveDB-Spearman top-12 SAE-layer worst-rank ensemble. The gamma grid starts at 1.0, two orders above the earlier 0.01 minimum, and uses 60 log-spaced steps.

In [ ]:
BEST_RANK_GAMMA_COLLECTION = f"best_mavedb_spearman_top{COLLECTION_TOP_N}"
BEST_RANK_GAMMA_AGGREGATION = "rank_worst"
BEST_RANK_GAMMA_MIN = 0.0001
BEST_RANK_GAMMA_MAX = 1
BEST_RANK_GAMMA_STEPS = 20
BEST_RANK_GAMMA_VALUES = np.logspace(
    np.log10(BEST_RANK_GAMMA_MIN),
    np.log10(BEST_RANK_GAMMA_MAX),
    BEST_RANK_GAMMA_STEPS,
)

CREATE_BEST_RANK_GAMMA_SWEEP_JOB = True
SUBMIT_BEST_RANK_GAMMA_SWEEP_JOB = False
RUN_BEST_RANK_GAMMA_SWEEP_LOCALLY = False
BEST_RANK_GAMMA_SWEEP_MEM = "96G"
BEST_RANK_GAMMA_SWEEP_TIME = "18:00:00"

assert BEST_RANK_GAMMA_COLLECTION in collections, f"Missing collection: {BEST_RANK_GAMMA_COLLECTION}"

best_rank_gamma_run_name = f"{BEST_RANK_GAMMA_COLLECTION}_worst_rank_gamma_sweep"
best_rank_gamma_dir = OUTPUT_ROOT / best_rank_gamma_run_name
best_rank_gamma_dir.mkdir(parents=True, exist_ok=True)
best_rank_gamma_log_dir = JOB_ROOT / "logs_best_rank_gamma_sweep"
best_rank_gamma_log_dir.mkdir(parents=True, exist_ok=True)

best_rank_gamma_metrics_path = best_rank_gamma_dir / f"{best_rank_gamma_run_name}_metrics.csv"
best_rank_gamma_component_consistency_path = best_rank_gamma_dir / f"{best_rank_gamma_run_name}_component_consistency.csv"
best_rank_gamma_figure_path = paths.figure_dir / "BRCA1_best_mavedb_spearman_top12_worst_rank_gamma_spearman_auc.png"
best_rank_gamma_payload_path = JOB_ROOT / f"{best_rank_gamma_run_name}_payload.pkl"
best_rank_gamma_script_path = JOB_ROOT / f"submit_{best_rank_gamma_run_name}.sh"
best_rank_gamma_worker_path = REPO_ROOT / "job_scripts" / "run_rank_ensemble_gamma_sweep.py"

best_rank_gamma_payload = {
    "repo_root": str(REPO_ROOT),
    "dataset_name": DATASET_NAME,
    "embedding_type": EMBEDDING_TYPE,
    "metrics_path": str(paths.spearman_auc_table_path),
    "sae_metrics_path": str(paths.fixed_sae_metrics_path),
    "clinvar_annotation_cache_path": str(paths.clinvar_annotation_cache_path),
    "primary_key": "hgvs_nt",
    "score_col": "score",
    "collection_top_n": COLLECTION_TOP_N,
    "balanced_per_model": BALANCED_PER_MODEL,
    "collection_name": BEST_RANK_GAMMA_COLLECTION,
    "aggregation": BEST_RANK_GAMMA_AGGREGATION,
    "gamma_values": BEST_RANK_GAMMA_VALUES,
    "min_review_stars": 0,
    "output_dir": str(best_rank_gamma_dir),
    "metrics_output_path": str(best_rank_gamma_metrics_path),
    "component_consistency_output_path": str(best_rank_gamma_component_consistency_path),
    "figure_output_path": str(best_rank_gamma_figure_path),
    "plot_title": "Best MaveDB Spearman top-12 worst-rank ensemble gamma sweep",
    "figure_dpi": FIGURE_DPI,
    "run_label": best_rank_gamma_run_name,
}

if CREATE_BEST_RANK_GAMMA_SWEEP_JOB:
    with best_rank_gamma_payload_path.open("wb") as handle:
        pickle.dump(best_rank_gamma_payload, handle)
    script_lines = [
        "#!/bin/bash",
        "#SBATCH --job-name=brca1_best_wr_gamma",
        f"#SBATCH -p {SLURM_PARTITION}",
        f"#SBATCH --cpus-per-task={SLURM_CPUS_PER_TASK}",
        f"#SBATCH --mem={BEST_RANK_GAMMA_SWEEP_MEM}",
        f"#SBATCH --time={BEST_RANK_GAMMA_SWEEP_TIME}",
        f"#SBATCH --output={best_rank_gamma_log_dir}/%x-%j.out",
        f"#SBATCH --error={best_rank_gamma_log_dir}/%x-%j.err",
        "",
        "set -euo pipefail",
        f"cd {REPO_ROOT}",
        f"{PYTHON_EXECUTABLE} {best_rank_gamma_worker_path} {best_rank_gamma_payload_path}",
    ]
    best_rank_gamma_script_path.write_text("\n".join(script_lines) + "\n")
    best_rank_gamma_script_path.chmod(0o755)
    print(f"Gamma sweep grid: {BEST_RANK_GAMMA_VALUES[0]:g} to {BEST_RANK_GAMMA_VALUES[-1]:g} across {len(BEST_RANK_GAMMA_VALUES)} steps")
    print(f"Gamma sweep payload: {best_rank_gamma_payload_path}")
    print(f"Gamma sweep Slurm script: {best_rank_gamma_script_path}")
    print(f"Submit with: sbatch {best_rank_gamma_script_path}")
    print(f"Expected metrics: {best_rank_gamma_metrics_path}")
    print(f"Expected figure: {best_rank_gamma_figure_path}")
    if SUBMIT_BEST_RANK_GAMMA_SWEEP_JOB:
        completed = subprocess.run(["sbatch", str(best_rank_gamma_script_path)], check=True, capture_output=True, text=True)
        print(completed.stdout.strip())

if RUN_BEST_RANK_GAMMA_SWEEP_LOCALLY:
    subprocess.run([PYTHON_EXECUTABLE, str(best_rank_gamma_worker_path), str(best_rank_gamma_payload_path)], check=True)

In [ ]:
if best_rank_gamma_metrics_path.is_file():
    best_rank_gamma_metrics_df = pd.read_csv(best_rank_gamma_metrics_path)
    best_rank_gamma_metrics_df["mavedb_spearman_rank"] = pd.to_numeric(best_rank_gamma_metrics_df["spearman_rho"], errors="coerce").rank(ascending=False, method="min")
    best_rank_gamma_metrics_df["auc_rank"] = pd.to_numeric(best_rank_gamma_metrics_df["auc"], errors="coerce").rank(ascending=False, method="min")
    best_rank_gamma_metrics_df["spearman_auc_rank_score"] = best_rank_gamma_metrics_df[["mavedb_spearman_rank", "auc_rank"]].mean(axis=1)
    best_rank_gamma_metrics_df = best_rank_gamma_metrics_df.sort_values(["spearman_auc_rank_score", "mavedb_spearman_rank", "auc_rank", "gamma"]).reset_index(drop=True)
    best_rank_gamma_summary_path = paths.table_dir / "BRCA1_best_mavedb_spearman_top12_worst_rank_gamma_sweep_summary.csv"
    best_rank_gamma_metrics_df.to_csv(best_rank_gamma_summary_path, index=False)

    display_columns = [
        "gamma",
        "spearman_auc_rank_score",
        "mavedb_spearman_rank",
        "auc_rank",
        "spearman_rho",
        "auc",
        "mean_component_rep_consistency",
        "n_variants",
        "n_score_sequences",
    ]
    display(
        best_rank_gamma_metrics_df[[col for col in display_columns if col in best_rank_gamma_metrics_df.columns]].head(15).assign(
            gamma=lambda df: pd.to_numeric(df["gamma"], errors="coerce").map(lambda x: f"{x:.4g}"),
            spearman_auc_rank_score=lambda df: pd.to_numeric(df["spearman_auc_rank_score"], errors="coerce").round(2),
            spearman_rho=lambda df: pd.to_numeric(df["spearman_rho"], errors="coerce").round(3),
            auc=lambda df: pd.to_numeric(df["auc"], errors="coerce").round(3),
            mean_component_rep_consistency=lambda df: pd.to_numeric(df["mean_component_rep_consistency"], errors="coerce").round(3),
        )
    )

    gamma_scatter_fig = h.plot_gamma_spearman_auc_scatter(
        best_rank_gamma_metrics_df,
        best_rank_gamma_figure_path,
        "Best MaveDB Spearman top-12 worst-rank ensemble: gamma sweep",
        figure_dpi=FIGURE_DPI,
    )
    if gamma_scatter_fig is not None:
        plt.show()
    print(f"Gamma sweep summary table: {best_rank_gamma_summary_path}")
    print(f"Gamma sweep scatter figure: {best_rank_gamma_figure_path}")
    print(f"Component consistency by gamma: {best_rank_gamma_component_consistency_path}")
else:
    best_rank_gamma_metrics_df = pd.DataFrame()
    print("No gamma sweep metrics found yet.")
    print(f"Run: sbatch {best_rank_gamma_script_path}")
    print(f"Expected metrics: {best_rank_gamma_metrics_path}")

## PCA Variance Breakdown For The Best MaveDB-Spearman Top-12 PCA Concat Model

This summarizes the selected PCA components per SAE layer and the variance explained by those selected components. If both gamma-objective runs exist, the MaveDB-gamma component table is preferred; the PCA component choices are independent of gamma selection.

In [ ]:
PCA_VARIANCE_MODEL_FILTER = f"best_mavedb_spearman_top{COLLECTION_TOP_N}"
PCA_VARIANCE_OBJECTIVE_PRIORITY = {"mavedb_spearman": 0, "cross_replicate_consistency": 1}

pca_variance_component_path = None
pca_variance_objective = None

if not esmc_ensemble_metrics_df.empty and "component_table_path" in esmc_ensemble_metrics_df.columns:
    pca_variance_candidates = esmc_ensemble_metrics_df[
        esmc_ensemble_metrics_df["aggregation"].eq("pca_concat")
        & esmc_ensemble_metrics_df["model_filter"].eq(PCA_VARIANCE_MODEL_FILTER)
        & esmc_ensemble_metrics_df["component_table_path"].notna()
    ].copy()
    if not pca_variance_candidates.empty:
        if "gamma_selection_objective" not in pca_variance_candidates.columns:
            pca_variance_candidates["gamma_selection_objective"] = "cross_replicate_consistency"
        pca_variance_candidates["_objective_priority"] = pca_variance_candidates["gamma_selection_objective"].map(PCA_VARIANCE_OBJECTIVE_PRIORITY).fillna(99)
        pca_variance_candidates = pca_variance_candidates.sort_values(["_objective_priority", "gamma_selection_objective"])
        for _, candidate_row in pca_variance_candidates.iterrows():
            candidate_path = Path(candidate_row["component_table_path"])
            if candidate_path.is_file():
                pca_variance_component_path = candidate_path
                pca_variance_objective = candidate_row.get("gamma_selection_objective", "")
                break

if pca_variance_component_path is None:
    globbed_component_paths = sorted(
        (OUTPUT_ROOT / "pca_concat").glob(f"**/pca_concat__{PCA_VARIANCE_MODEL_FILTER}*_pca_components.csv"),
        key=lambda path: (0 if "__mavedb_spearman" in path.name else 1, str(path)),
    )
    for candidate_path in globbed_component_paths:
        if candidate_path.is_file():
            pca_variance_component_path = candidate_path
            pca_variance_objective = "mavedb_spearman" if "__mavedb_spearman" in candidate_path.name else "cross_replicate_consistency"
            break

if pca_variance_component_path is None:
    pca_variance_summary_df = pd.DataFrame()
    print(f"No PCA component table found for {PCA_VARIANCE_MODEL_FILTER}.")
else:
    pca_components_df = pd.read_csv(pca_variance_component_path)
    selected_flag = pca_components_df["selected"]
    if selected_flag.dtype == bool:
        selected_mask = selected_flag
    else:
        selected_mask = selected_flag.astype(str).str.lower().isin(["true", "1", "yes"])
    selected_pca_components_df = pca_components_df[selected_mask].copy()
    pca_components_df["layer"] = pd.to_numeric(pca_components_df["layer"], errors="coerce")
    selected_pca_components_df["layer"] = pd.to_numeric(selected_pca_components_df["layer"], errors="coerce")

    group_cols = ["model_short", "model", "layer", "model_label"]
    selected_summary = selected_pca_components_df.groupby(group_cols, dropna=False).agg(
        selected_pcs=("pc_idx", "count"),
        selected_explained_variance_ratio=("explained_variance_ratio", "sum"),
        mean_selected_explained_variance_ratio=("explained_variance_ratio", "mean"),
        max_selected_explained_variance_ratio=("explained_variance_ratio", "max"),
        first_selected_concat_component=("concat_component_idx", "min"),
        last_selected_concat_component=("concat_component_idx", "max"),
    ).reset_index()
    available_summary = pca_components_df.groupby(group_cols, dropna=False).agg(
        available_pcs_in_component_table=("pc_idx", "count"),
        available_explained_variance_ratio=("explained_variance_ratio", "sum"),
    ).reset_index()
    pca_variance_summary_df = selected_summary.merge(available_summary, on=group_cols, how="left")
    pca_variance_summary_df["selected_fraction_of_table_variance"] = (
        pca_variance_summary_df["selected_explained_variance_ratio"] / pca_variance_summary_df["available_explained_variance_ratio"]
    )
    pca_variance_summary_df = pca_variance_summary_df.sort_values(["model_short", "layer"]).reset_index(drop=True)

    pca_variance_summary_path = paths.table_dir / "BRCA1_best_mavedb_spearman_top12_pca_concat_selected_pca_variance_by_layer.csv"
    pca_variance_selected_components_path = paths.table_dir / "BRCA1_best_mavedb_spearman_top12_pca_concat_selected_pca_components.csv"
    pca_variance_figure_path = paths.figure_dir / "BRCA1_best_mavedb_spearman_top12_pca_concat_selected_pca_variance_by_layer.png"
    pca_variance_summary_df.to_csv(pca_variance_summary_path, index=False)
    selected_pca_components_df.to_csv(pca_variance_selected_components_path, index=False)

    display(
        pca_variance_summary_df.assign(
            layer=lambda df: pd.to_numeric(df["layer"], errors="coerce").astype("Int64"),
            selected_explained_variance_ratio=lambda df: pd.to_numeric(df["selected_explained_variance_ratio"], errors="coerce").round(4),
            available_explained_variance_ratio=lambda df: pd.to_numeric(df["available_explained_variance_ratio"], errors="coerce").round(4),
            selected_fraction_of_table_variance=lambda df: pd.to_numeric(df["selected_fraction_of_table_variance"], errors="coerce").round(3),
            mean_selected_explained_variance_ratio=lambda df: pd.to_numeric(df["mean_selected_explained_variance_ratio"], errors="coerce").round(4),
            max_selected_explained_variance_ratio=lambda df: pd.to_numeric(df["max_selected_explained_variance_ratio"], errors="coerce").round(4),
        )
    )

    plot_df = pca_variance_summary_df.copy()
    plot_df["layer_label"] = plot_df["model_short"].astype(str) + " L" + pd.to_numeric(plot_df["layer"], errors="coerce").astype("Int64").astype(str)
    plot_df = plot_df.sort_values("selected_explained_variance_ratio", ascending=True)
    fig, ax = plt.subplots(figsize=(9.0, max(4.0, 0.38 * len(plot_df))))
    sns.barplot(
        data=plot_df,
        y="layer_label",
        x="selected_explained_variance_ratio",
        hue="model_short",
        dodge=False,
        ax=ax,
    )
    ax.set_xlabel("Sum of selected PCA explained variance ratio")
    ax.set_ylabel("SAE layer")
    ax.set_title("Selected PCA variance by SAE layer")
    ax.legend(title="Model", loc="lower right")
    fig.tight_layout()
    fig.savefig(pca_variance_figure_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()

    print(f"PCA component table used: {pca_variance_component_path}")
    print(f"Gamma objective for selected component table: {pca_variance_objective}")
    print(f"PCA variance summary table: {pca_variance_summary_path}")
    print(f"Selected PCA components table: {pca_variance_selected_components_path}")
    print(f"PCA variance figure: {pca_variance_figure_path}")

## Plot Against Previous Layer-Level Results

This plot overlays the new ESM-C ensemble methods onto the existing layer-level Spearman/AUC results from `lower_context.ipynb`.

In [ ]:
if not esmc_ensemble_metrics_df.empty:
    previous_df = h.load_previous_metrics_for_combined_plot(
        all_binary_table_path=paths.spearman_auc_table_path,
        table_dir=paths.table_dir,
        star_combined_table_path=paths.spearman_auc_star_combined_table_path,
    )
    spearman_auc_plot_rows_path = paths.table_dir / "BRCA1_esmc_sae_layer_ensemble_spearman_auc_plot_rows.csv"
    esmc_ensemble_metrics_df.to_csv(spearman_auc_plot_rows_path, index=False)

    marker_cycle = ["P", "X", "D", "s", "^", "v", "o"]
    color_cycle = sns.color_palette("tab20", n_colors=max(20, esmc_ensemble_metrics_df["method_label"].nunique()))
    marker_specs = {}
    for idx, method_label in enumerate(esmc_ensemble_metrics_df["method_label"].drop_duplicates()):
        is_mavedb_gamma = "MaveDB gamma" in method_label
        marker_specs[method_label] = {
            "marker": "X" if is_mavedb_gamma else marker_cycle[idx % len(marker_cycle)],
            "size": 145 if "PCA" in method_label else 115,
            "color": color_cycle[idx % len(color_cycle)],
            "offset": (6, -9),
        }

    combined_figure_path = paths.figure_dir / "BRCA1_esmc_sae_layer_ensemble_with_previous_spearman_auc_by_review_stars.png"
    fig = h.plot_ensemble_with_previous(
        previous_df,
        esmc_ensemble_metrics_df,
        combined_figure_path,
        "ESM-C SAE layer ensembles over previous Spearman/AUC comparison",
        marker_specs,
        figure_dpi=FIGURE_DPI,
        review_star_thresholds=[1, 2, 3],
        annotate_points=False,
        include_legend=False,
    )
    plt.show()

    legend_figure_path = paths.figure_dir / "BRCA1_esmc_sae_layer_ensemble_spearman_auc_legend.png"
    legend_fig = h.plot_ensemble_comparison_legend(
        marker_specs,
        legend_figure_path,
        figure_dpi=FIGURE_DPI,
        ncol=3,
    )
    plt.show()

    display(
        esmc_ensemble_metrics_df[
            ["review_filter", "method_label", "gamma_selection_objective", "model_filter", "spearman_rho", "auc", "n_variants"]
        ].assign(
            spearman_rho=lambda df: pd.to_numeric(df["spearman_rho"], errors="coerce").round(3),
            auc=lambda df: pd.to_numeric(df["auc"], errors="coerce").round(3),
        )
    )
    print(f"Combined figure: {combined_figure_path}")
    print(f"Separate legend figure: {legend_figure_path}")
    print(f"Spearman/AUC plot rows table: {spearman_auc_plot_rows_path}")
else:
    print("No ensemble metrics are available yet. Run or collect the ensemble task outputs first.")

## Gamma Consistency Plot Locations

Every PCA-concat task saves its cross-replicate consistency by gamma plot as `*_gamma_consistency.png`. The table below lists those exact paths.

In [ ]:
if not esmc_ensemble_metrics_df.empty and "gamma_plot_path" in esmc_ensemble_metrics_df.columns:
    gamma_columns = ["method_label", "model_filter", "gamma_opt", "gamma_plot_path"]
    if "gamma_table_path" in esmc_ensemble_metrics_df.columns:
        gamma_columns.append("gamma_table_path")
    gamma_plot_df = (
        esmc_ensemble_metrics_df.dropna(subset=["gamma_plot_path"])
        .loc[lambda df: df["gamma_plot_path"].astype(str).ne("")]
        [gamma_columns]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    """display(gamma_plot_df)
    for _, row in gamma_plot_df.iterrows():
        plot_path = Path(row["gamma_plot_path"])
        display(Markdown(f"**{row['method_label']}**  \n`{plot_path}`"))
        if plot_path.is_file():
            display(Image(filename=str(plot_path), width=780))
        else:
            print(f"Missing gamma plot: {plot_path}")"""
print(f"PCA gamma consistency directory: {OUTPUT_ROOT / 'pca_concat'}")


## Task Status Audit

In [ ]:
status_files = sorted((OUTPUT_ROOT / "status").glob("*_status.json"))
status_rows = []
for status_path in status_files:
    with status_path.open() as handle:
        row = __import__("json").load(handle)
    row["status_path"] = str(status_path)
    status_rows.append(row)
status_df = pd.DataFrame(status_rows)
if not status_df.empty:
    display(status_df[[col for col in ["task_idx", "status", "task_type", "collection_name", "aggregation", "n_sources", "n_selected_components", "gamma_plot_path", "error", "status_path"] if col in status_df.columns]])
else:
    print(f"No status files found yet under {OUTPUT_ROOT / 'status'}")

## Best Model Fitness Distribution By ClinVar Review Stars

Distribution of the top-ranked ensemble's fitness (rank) scores for ClinVar pathogenic vs. benign variants, paneled by review-star threshold. Pathogenic and benign histograms share bins and are drawn translucent so their overlap is visible, and each panel reports the corresponding AUC.

In [ ]:
# Best-model fitness-score distribution split by ClinVar pathogenic/benign across review-star thresholds.
if not performance_summary_df.empty:
    best_model_row = performance_summary_df.sort_values("spearman_auc_rank").iloc[0]
    best_model_label = best_model_row["method_label"]
    best_model_fitness_path = h.as_existing_path(best_model_row.get("fitness_path", ""))

    if best_model_fitness_path is None:
        print(f"No fitness table found for best model {best_model_label!r}; cannot plot score distributions.")
    else:
        best_fitness_df = pd.read_csv(best_model_fitness_path)[["SequenceIndex", "fitness"]].copy()
        best_fitness_df["SequenceIndex"] = best_fitness_df["SequenceIndex"].astype(str)
        best_fitness_df["fitness"] = pd.to_numeric(best_fitness_df["fitness"], errors="coerce")
        best_fitness_df = best_fitness_df.dropna(subset=["fitness"])

        pathogenicity_palette = {"benign": "#2c7fb8", "pathogenic": "#d7301f"}
        star_panels = []
        for min_review_stars, review_filter in h.review_star_specs():
            annotation_map = h.clinvar_binary_annotation_map(
                paths.clinvar_annotation_cache_path,
                primary_key="hgvs_nt",
                min_review_stars=min_review_stars,
            )
            panel_df = best_fitness_df.copy()
            panel_df["annotation"] = panel_df["SequenceIndex"].map(annotation_map)
            panel_df = panel_df[panel_df["annotation"].isin(["benign", "pathogenic"])].dropna(subset=["fitness"])
            if panel_df.empty:
                continue
            panel_auc = h.classification_metrics_for_fitness(panel_df)["auc"]
            star_panels.append((min_review_stars, review_filter, panel_df, panel_auc))

        if not star_panels:
            print("No pathogenic/benign annotated variants overlap the best-model fitness scores.")
        else:
            # Shared bins so the translucent pathogenic/benign overlap is comparable across panels.
            all_fitness = best_fitness_df["fitness"].to_numpy(dtype=float)
            bin_edges = np.linspace(np.nanmin(all_fitness), np.nanmax(all_fitness), 31)

            ncols = min(len(star_panels), 2)
            nrows = int(np.ceil(len(star_panels) / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 4.0 * nrows), squeeze=False)
            axes_flat = axes.ravel()
            for ax, (min_review_stars, review_filter, panel_df, panel_auc) in zip(axes_flat, star_panels):
                for annotation in ["benign", "pathogenic"]:
                    values = panel_df.loc[panel_df["annotation"].eq(annotation), "fitness"].to_numpy(dtype=float)
                    ax.hist(
                        values,
                        bins=bin_edges,
                        color=pathogenicity_palette[annotation],
                        alpha=0.5,
                        label=f"{annotation} (n={len(values)})",
                        edgecolor="white",
                        linewidth=0.4,
                    )
                ax.set_title(f"{review_filter} \u2014 AUC = {panel_auc:.3f}")
                ax.set_xlabel("Fitness (rank)")
                ax.set_ylabel("Frequency")
                ax.legend(title="ClinVar annotation", fontsize=8)
            for ax in axes_flat[len(star_panels):]:
                ax.axis("off")
            fig.suptitle(
                f"Best model fitness distribution split by ClinVar annotation\n{best_model_label}",
                y=1.0,
            )
            fig.tight_layout()
            best_model_distribution_figure_path = paths.figure_dir / "BRCA1_best_model_fitness_distribution_by_review_stars.png"
            fig.savefig(best_model_distribution_figure_path, dpi=FIGURE_DPI, bbox_inches="tight")
            plt.show()
            print(f"Best model (Spearman/AUC rank 1): {best_model_label}")
            print(f"Best model fitness distribution figure: {best_model_distribution_figure_path}")
else:
    print("No performance summary is available yet. Run or collect the ensemble task outputs first.")
